# Chapter 3 — Coding Attention Mechanisms

Implementing self-attention: computing attention scores between a query token and all input tokens to build a context vector.

## Figure 3.8 — Attention scores for the context vector

![Figure 3.8: computing attention scores between the query x^(2) and all input elements as dot products](1.png)

*The overall goal is to illustrate the computation of the context vector $z^{(2)}$ using the second input element, $x^{(2)}$, as a query. This figure shows the first intermediate step: computing the attention scores $\omega$ between the query $x^{(2)}$ and all other input elements as a dot product. (The numbers are truncated to one digit after the decimal point to reduce visual clutter.)*

## Input embeddings

Each row is the 3-dimensional embedding of one token in the sentence *"Your journey starts with one step"* ($x^{(1)}$ through $x^{(6)}$).

In [1]:
import torch
import torch.nn as nn

# 6 tokens (one per word), each a 6-dimensional embedding -> shape [6, 6]
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89, 0.21, 0.67, 0.05],  # Your     (x^1)
        [0.55, 0.87, 0.66, 0.34, 0.12, 0.78],  # journey  (x^2)
        [0.57, 0.85, 0.64, 0.29, 0.18, 0.71],  # starts   (x^3)
        [0.22, 0.58, 0.33, 0.91, 0.44, 0.26],  # with     (x^4)
        [0.77, 0.25, 0.10, 0.48, 0.83, 0.52],  # one      (x^5)
        [0.05, 0.80, 0.55, 0.16, 0.39, 0.94],  # step     (x^6)
    ]
)
print("inputs shape:", inputs.shape)

query = inputs[1]
attention_scores = torch.zeros(inputs.shape[0])
for i in range(len(inputs)):
    attention_weights = torch.dot(query, inputs[i])
    attention_scores[i] = attention_weights

print(attention_scores) # -> then we normalize the attention scores  ( softmax)
# idea is to make add a learnable parameter to the attention weights

# shortcut

attnetion_scores = inputs @ inputs.T
print(attnetion_scores)
attention_weights = torch.softmax(attnetion_scores, dim=1)
print("normalized attention weights")
print(attention_weights)

inputs shape: torch.Size([6, 6])
tensor([1.1452, 2.2334, 2.1494, 1.4084, 1.3754, 1.9209])
tensor([[1.4950, 1.1452, 1.1592, 0.9742, 1.1405, 0.9729],
        [1.1452, 2.2334, 2.1494, 1.4084, 1.3754, 1.9209],
        [1.1592, 2.1494, 2.0776, 1.3573, 1.3732, 1.8445],
        [0.9742, 1.4084, 1.3573, 1.5830, 1.2846, 1.2181],
        [1.1405, 1.3754, 1.3732, 1.2846, 1.8551, 1.1828],
        [0.9729, 1.9209, 1.8445, 1.2181, 1.1828, 2.0063]])
normalized attention weights
tensor([[0.2321, 0.1636, 0.1659, 0.1379, 0.1628, 0.1377],
        [0.0875, 0.2597, 0.2388, 0.1138, 0.1101, 0.1900],
        [0.0940, 0.2530, 0.2355, 0.1146, 0.1164, 0.1865],
        [0.1178, 0.1819, 0.1728, 0.2165, 0.1607, 0.1503],
        [0.1287, 0.1628, 0.1625, 0.1487, 0.2630, 0.1343],
        [0.0885, 0.2285, 0.2117, 0.1132, 0.1092, 0.2489]])


In [2]:
## Extend it with learnable weights !! yayy
## Using nn.Linear for the Q/K/V projections (better weight init, matches GPT-2 layout)
class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)
        attn_scores = Q @ K.T
        # scale by sqrt(d_out): divide the scores, don't sqrt the scores
        attn_weights = torch.softmax(attn_scores / self.d_out**0.5, dim=-1)
        context_vector = attn_weights @ V
        return context_vector


sa = SelfAttention(d_in=6, d_out=2)
print(sa(inputs))


tensor([[-0.2869,  0.3030],
        [-0.2897,  0.2990],
        [-0.2894,  0.2994],
        [-0.2883,  0.3001],
        [-0.2869,  0.3033],
        [-0.2914,  0.2973]], grad_fn=<MmBackward0>)


### Causal attention, also known as masked attention

![](2.png)

In causal attention, we mask out the attention weights above the diagonal such that for
a given input, the LLM can’t access future tokens when computing the context vectors using the
attention weights. For example, for the word “journey” in the second row, we only keep the attention
weights for the words before (“Your”) and in the current position (“journey”).

In [3]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

    def forward(self, x):
        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)

        attn_scores = Q @ K.T
        # Option B: set future positions to -inf BEFORE softmax,
        # so they become exactly 0 after softmax (no renormalization needed).
        attn_scores = attn_scores.masked_fill(self.mask.bool(), -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.d_out**0.5, dim=-1)
        context_vector = attn_weights @ V
        return context_vector


torch.manual_seed(123)
context_length = inputs.shape[0]
ca = CausalAttention(d_in=6, d_out=6, context_length=context_length)
print(ca(inputs))


tensor([[ 0.0713,  0.2563, -0.1121, -0.0174,  0.6784, -0.0869],
        [-0.2329,  0.2960, -0.1367,  0.1354,  0.5706, -0.0424],
        [-0.3427,  0.3107, -0.1495,  0.1822,  0.5406, -0.0283],
        [-0.3267,  0.2998, -0.0475,  0.2416,  0.4763, -0.0574],
        [-0.3297,  0.2602,  0.0055,  0.1626,  0.4735, -0.0158],
        [-0.3459,  0.2698,  0.0082,  0.1920,  0.4374,  0.0012]],
       grad_fn=<MmBackward0>)


### Adding dropout

Dropout randomly zeroes out a fraction of the **attention weights** during training, which prevents the model from over-relying on any single token (regularization). It is applied **after** masking + softmax. PyTorch automatically scales the surviving weights by `1/(1 - p)` so the expected sum is preserved. Dropout is only active in **training mode** (`model.train()`); it's disabled in `model.eval()`.

In [4]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1),
        )

    def forward(self, x):
        num_tokens = x.shape[0]
        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)

        attn_scores = Q @ K.T
        # 1) mask future positions with -inf before softmax
        attn_scores = attn_scores.masked_fill(
            self.mask[:num_tokens, :num_tokens].bool(), -torch.inf
        )
        # 2) normalize with scaled softmax
        attn_weights = torch.softmax(attn_scores / self.d_out**0.5, dim=-1)
        # 3) drop a fraction of the attention weights (regularization)
        attn_weights = self.dropout(attn_weights)

        context_vector = attn_weights @ V
        return context_vector


torch.manual_seed(123)
context_length = inputs.shape[0]
ca = CausalAttention(d_in=6, d_out=6, context_length=context_length, dropout=0.5)

ca.train()  # dropout active
print("train mode (dropout on):")
print(ca(inputs))

ca.eval()  # dropout disabled
print("\neval mode (dropout off):")
print(ca(inputs))


train mode (dropout on):
tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-0.3242,  0.4099, -0.1893,  0.1884,  0.7891, -0.0584],
        [-0.3987,  0.3078,  0.0403,  0.3372,  0.4071, -0.0776],
        [-0.4477,  0.3965,  0.0753,  0.2221,  0.7758, -0.0326],
        [-0.3660,  0.2471,  0.1571,  0.2110,  0.3681,  0.0285]],
       grad_fn=<MmBackward0>)

eval mode (dropout off):
tensor([[ 0.0713,  0.2563, -0.1121, -0.0174,  0.6784, -0.0869],
        [-0.2329,  0.2960, -0.1367,  0.1354,  0.5706, -0.0424],
        [-0.3427,  0.3107, -0.1495,  0.1822,  0.5406, -0.0283],
        [-0.3267,  0.2998, -0.0475,  0.2416,  0.4763, -0.0574],
        [-0.3297,  0.2602,  0.0055,  0.1626,  0.4735, -0.0158],
        [-0.3459,  0.2698,  0.0082,  0.1920,  0.4374,  0.0012]],
       grad_fn=<MmBackward0>)


In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.heads = nn.ModuleList(
            [
                CausalAttention(
                    d_in=d_in,
                    d_out=self.head_dim,
                    context_length=context_length,
                    dropout=dropout,
                    qkv_bias=qkv_bias,
                )
                for _ in range(num_heads)
            ]
        )
    def forward(self,x  ):
        return torch.cat([head(x) for head in self.heads], dim=-1)

## Efficient multi-head attention (weight split + batched heads)

The wrapper version above runs each head in a Python `for` loop — correct but slow. The efficient version uses **one** set of large `W_query/W_key/W_value` projections, then reshapes the output into separate heads so all heads are computed **in parallel** as a single batched matmul.

The shape choreography:
1. Project `x` `[b, T, d_in]` → `Q/K/V` `[b, T, d_out]`
2. `view` to split the last dim into heads → `[b, T, num_heads, head_dim]`
3. `transpose(1, 2)` to bring heads next to the batch dim → `[b, num_heads, T, head_dim]` (now each head is just another batch dimension)
4. Attention math runs on all heads at once → context `[b, num_heads, T, head_dim]`
5. `transpose` back + `view` to merge heads → `[b, T, d_out]`
6. A final `out_proj` linear mixes information across heads.

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, dropout, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # per-head dimension

        # One big projection each; heads are carved out of d_out later.
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # mixes head outputs together
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1),
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        Q = self.W_query(x)  # [b, T, d_out]
        K = self.W_key(x)
        V = self.W_value(x)

        # Split d_out -> (num_heads, head_dim): [b, T, num_heads, head_dim]
        Q = Q.view(b, num_tokens, self.num_heads, self.head_dim)
        K = K.view(b, num_tokens, self.num_heads, self.head_dim)
        V = V.view(b, num_tokens, self.num_heads, self.head_dim)

        # Bring heads beside batch: [b, num_heads, T, head_dim]
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        # Batched attention over all heads at once: [b, num_heads, T, T]
        attn_scores = Q @ K.transpose(2, 3)
        mask_bool = self.mask[:num_tokens, :num_tokens].bool()
        attn_scores = attn_scores.masked_fill(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Weighted sum of values: [b, num_heads, T, head_dim] -> [b, T, num_heads, head_dim]
        context = (attn_weights @ V).transpose(1, 2)
        # Merge heads back into d_out: [b, T, d_out]
        context = context.contiguous().view(b, num_tokens, self.d_out)
        context = self.out_proj(context)
        return context


torch.manual_seed(123)
# Make a batch of 2 sequences from the 2D inputs: [2, 6, 6]
batch = torch.stack((inputs, inputs), dim=0)
print("batch shape:", batch.shape)

mha = MultiHeadAttention(
    d_in=6,
    d_out=6,
    num_heads=2,
    context_length=batch.shape[1],
    dropout=0.0,
)
context_vecs = mha(batch)
print("output shape:", context_vecs.shape)
print(context_vecs)


batch shape: torch.Size([2, 6, 6])
output shape: torch.Size([2, 6, 6])
tensor([[[ 1.9300e-01, -7.6635e-02,  1.7446e-02, -5.4641e-02, -2.4626e-01,
          -2.8865e-01],
         [ 6.2797e-02, -3.5613e-02, -2.6215e-02, -3.4476e-02, -3.1846e-01,
          -3.2146e-01],
         [ 2.0959e-02, -2.0405e-02, -3.8418e-02, -3.1234e-02, -3.3949e-01,
          -3.3139e-01],
         [ 2.9233e-02, -6.9246e-02, -3.4394e-02,  9.4565e-03, -3.7427e-01,
          -3.5567e-01],
         [ 5.7400e-02, -7.5145e-02, -1.4824e-02, -1.2870e-02, -3.4476e-01,
          -3.2437e-01],
         [ 3.9361e-02, -7.1521e-02, -2.9318e-02,  3.1479e-05, -3.6120e-01,
          -3.2807e-01]],

        [[ 1.9300e-01, -7.6635e-02,  1.7446e-02, -5.4641e-02, -2.4626e-01,
          -2.8865e-01],
         [ 6.2797e-02, -3.5613e-02, -2.6215e-02, -3.4476e-02, -3.1846e-01,
          -3.2146e-01],
         [ 2.0959e-02, -2.0405e-02, -3.8418e-02, -3.1234e-02, -3.3949e-01,
          -3.3139e-01],
         [ 2.9233e-02, -6.9246e-02, 